## Init

In [4]:
import pandas as pd
import numpy as np

pd.options.mode.copy_on_write = True

## Load Dataset

In [5]:
folder = "C:/Users/B00955739/OneDrive - Ulster University/Documents/PhD/Results/Predicting_predictability/"
# folder = "C:/Users/kevin/OneDrive - Ulster University/Documents/PhD/Results/Predicting_predictability/"
filename = "main_run_v2.csv"
csv_file = folder + filename

In [6]:
raw_df = pd.read_csv(csv_file)

In [7]:
print("Read csv file {}".format(csv_file))
print("{} rows and {} columns".format(len(raw_df), len(raw_df.columns)))

Read csv file C:/Users/B00955739/OneDrive - Ulster University/Documents/PhD/Results/Predicting_predictability/main_run_v2.csv
372044 rows and 202 columns


## Correct Lyapunov

In [8]:
lyap_cols = []
for d in range(10):
    lyap_cols.append('lyap_{}'.format(d))

print(lyap_cols)

['lyap_0', 'lyap_1', 'lyap_2', 'lyap_3', 'lyap_4', 'lyap_5', 'lyap_6', 'lyap_7', 'lyap_8', 'lyap_9']


In [11]:
raw_df['largest_lyap'] = np.nanmax(raw_df[lyap_cols].values, axis=1)

In [12]:
raw_df['corrected'] = False
raw_df.loc[raw_df.lyap_0 != raw_df.largest_lyap, "corrected"] = True

In [23]:
raw_df['lyap_spectrum_full'] = raw_df[lyap_cols].values.tolist()

In [25]:
print(raw_df.lyap_spectrum_full[89])

[0.4494251786343348, 0.1159725337445912, nan, nan, nan, nan, nan, nan, nan, nan]


In [29]:
new_lyaps = np.empty((10, len(raw_df)))

row_count = 0
for row_idx, row in raw_df.iterrows():
    lyaps = np.array(row['lyap_spectrum_full'])
    sorted_lyaps = lyaps[np.argsort(-lyaps)]
    for d in range(10):
        if d >= row.dim:
            new_lyaps[d, row_count] = np.nan
        else:
            new_lyaps[d, row_count] = sorted_lyaps[d]
    row_count += 1

for d in range(10):
    raw_df['lyap_{}'.format(d)] = new_lyaps[d, :]

In [30]:
check = raw_df[raw_df.lyap_0 != raw_df.largest_lyap]
print(len(check))

0


In [31]:
changes = raw_df[raw_df.corrected]

In [32]:
changes.head()

,Unnamed: 0,dim,mat_type,epsilon,map_list,map_list_str,gamma,reverse,matrix,cml,...,lyap_3,lyap_4,lyap_5,lyap_6,lyap_7,lyap_8,lyap_9,largest_lyap,corrected,lyap_spectrum_full
74,74,2,NN,0.2,[<functions_v6.tent_map object at 0x000002990D...,"[('tent_map', {'a': 1.9999999999, 'b': 27.2990...",1.0,False,[[0.8 0.2]\n [0.2 0.8]],<functions_v6.CML object at 0x00000298D6CA4190>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3.748206,True,"[3.740968538739892, 3.7482058373940608, nan, n..."
78,78,2,NN,0.2,[<functions_v6.tent_map object at 0x000002990D...,"[('tent_map', {'a': 1.9999999999, 'b': 1}), ('...",3.0,True,[[0.8 0.2]\n [0.2 0.8]],<functions_v6.CML object at 0x00000298E6A58150>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0.442375,True,"[0.4330940592510075, 0.4423746780028106, nan, ..."
158,158,2,FC,0.3,[<functions_v6.sluze_map object at 0x000002990...,"[('sluze_map', {'m': 0.8, 'p': 0.2}), ('sin_ci...",0.5,True,[[0.7 0.3]\n [0.3 0.7]],<functions_v6.CML object at 0x000002990DB52890>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.546704,True,"[-0.5467074425743378, -0.5467042315750441, nan..."
178,178,2,RAND,0.3,[<functions_v6.sincircle_map object at 0x00000...,"[('sin_circle_map', {'omega': 1.41421356237309...",1.0,True,[[0.54290844 0.45709156]\n [0.27560978 0.72439...,<functions_v6.CML object at 0x000002990D9E7650>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.766139,True,"[-0.7661638445534827, -0.7661387362557616, nan..."
314,314,2,NN,0.6,[<functions_v6.sincircle_map object at 0x00000...,"[('sin_circle_map', {'omega': 0.5, 'k': 2}), (...",1.0,True,[[0.4 0.6]\n [0.6 0.4]],<functions_v6.CML object at 0x000002990DDC3890>,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,-0.111572,True,"[-0.1115717756947025, -0.1115717755954836, nan..."


In [33]:
print(len(changes))

11298


In [34]:
new_filename = "main_run_v2_corrected.csv"

raw_df.to_csv(folder + new_filename)